In [ ]:
!pip install pymoo pyswarm -q

In [ ]:
from __future__ import annotations
import os
import time
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.optimize import minimize
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.operators.crossover.hux import HUX
from pymoo.operators.mutation.bitflip import BitflipMutation
from pyswarm import pso


@dataclass
class Config:
    dataset_name: str = "census"
    classifier_name: str = "svm"

    random_state: int = 42
    alpha: float = 0.90
    ga_pop_size : int = 12
    ga_generations : int = 10
    
    pso_swarmsize : int = 12
    pso_maxiter : int = 10
    pso_omega: float = 0.5
    pso_phip: float = 1.5
    pso_phig: float = 1.5
    pso_threshold: float = 0.5
    sample_rows: Optional[int] = None


def get_classifier(name: str, random_state: int = 42):
    name = name.lower()

    if name == "svm":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", C=1.0, gamma="scale",max_iter=500))
        ])

    if name == "knn":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=5))
        ])

    if name == "dt":
        return Pipeline([
            ("clf", DecisionTreeClassifier(
                max_depth=6,
                min_samples_split=100,
                min_samples_leaf=50,
                random_state=random_state
            ))
        ])

    raise ValueError("classifier_name must be one of: 'svm', 'knn', 'dt'")

def infer_target_column(df: pd.DataFrame, dataset_name: str) -> str:
    cols = set(df.columns)

    if dataset_name == "santander":
        if "target" in cols:
            return "target"
        raise ValueError("For santander, 'target' column not found.")

    candidate_targets = [
        "income", "Income", "target", "Target", "class", "Class", "label"
    ]
    for c in candidate_targets:
        if c in cols:
            return c

    return df.columns[-1]


def load_census_from_kaggle(config: Config):
    possible_paths = [
        "/kaggle/input/datasets/tawfikelmetwally/census-income-dataset/adult.csv"
    ]

    csv_path = None
    for p in possible_paths:
        if os.path.exists(p):
            csv_path = p
            break

    if csv_path is None:
        raise FileNotFoundError(
            "Census CSV not found. Check file names in /kaggle/input/census-income-dataset/"
        )

    df = pd.read_csv(csv_path)

    if config.sample_rows is not None and len(df) > config.sample_rows:
        df = df.sample(config.sample_rows, random_state=config.random_state).reset_index(drop=True)

    target_col = infer_target_column(df, "census")

    X_df = df.drop(columns=[target_col]).copy()
    y = df[target_col].copy()
    for col in X_df.select_dtypes(include=["object"]).columns:
        X_df[col] = X_df[col].astype(str).str.strip()

    if y.dtype == "object":
        y = y.astype(str).str.strip()

    X_df = pd.get_dummies(X_df, drop_first=False)

    if y.dtype == "object":
        le = LabelEncoder()
        y = le.fit_transform(y)
    else:
        y = y.values

    X = X_df.values.astype(float)
    feature_names = X_df.columns.tolist()

    return X, y, feature_names, df


def load_santander_from_kaggle(config: Config):
    possible_paths = [
        "/kaggle/input/competitions/santander-customer-transaction-prediction/train.csv",
    ]

    csv_path = None
    for p in possible_paths:
        if os.path.exists(p):
            csv_path = p
            break

    if csv_path is None:
        raise FileNotFoundError(
            "Santander train.csv not found. Check file names in /kaggle/input/"
        )

    df = pd.read_csv(csv_path)

    if config.sample_rows is not None and len(df) > config.sample_rows:
        df = df.sample(config.sample_rows, random_state=config.random_state).reset_index(drop=True)

    target_col = infer_target_column(df, "santander")

    drop_cols = [target_col]
    if "ID_code" in df.columns:
        drop_cols.append("ID_code")

    X_df = df.drop(columns=drop_cols).copy()
    y = df[target_col].values

    X = X_df.values.astype(float)
    feature_names = X_df.columns.tolist()

    return X, y, feature_names, df


def load_dataset(config: Config):
    if config.dataset_name == "census":
        return load_census_from_kaggle(config)
    elif config.dataset_name == "santander":
        return load_santander_from_kaggle(config)
    else:
        raise ValueError("dataset_name must be 'census' or 'santander'")


class FeatureSelectionEvaluator:

    def __init__(
        self,
        X_train: np.ndarray,
        X_val: np.ndarray,
        y_train: np.ndarray,
        y_val: np.ndarray,
        classifier_name: str,
        alpha: float = 0.99,
        random_state: int = 42,
    ):
        self.X_train = X_train
        self.X_val = X_val
        self.y_train = y_train
        self.y_val = y_val
        self.n_features = X_train.shape[1]
        self.alpha = alpha
        self.model = get_classifier(classifier_name, random_state=random_state)
        self.cache: Dict[Tuple[int, ...], float] = {}

    def score_mask(self, mask: np.ndarray) -> float:
        mask = np.asarray(mask).astype(int).ravel()

        if mask.sum() == 0:
            return 1.0

        key = tuple(mask.tolist())
        if key in self.cache:
            return self.cache[key]

        X_tr = self.X_train[:, mask == 1]
        X_va = self.X_val[:, mask == 1]

        self.model.fit(X_tr, self.y_train)
        y_pred = self.model.predict(X_va)

        acc = accuracy_score(self.y_val, y_pred)
        feature_ratio = mask.sum() / self.n_features
        objective = self.alpha * (1.0 - acc) + (1.0 - self.alpha) * feature_ratio

        self.cache[key] = objective
        return objective

    def accuracy_of_mask(self, mask: np.ndarray) -> float:
        mask = np.asarray(mask).astype(int).ravel()

        if mask.sum() == 0:
            return 0.0

        X_tr = self.X_train[:, mask == 1]
        X_va = self.X_val[:, mask == 1]

        self.model.fit(X_tr, self.y_train)
        y_pred = self.model.predict(X_va)

        return accuracy_score(self.y_val, y_pred)


class GAFeatureSelectionProblem(ElementwiseProblem):
    def __init__(self, evaluator: FeatureSelectionEvaluator):
        super().__init__(
            n_var=evaluator.n_features,
            n_obj=1,
            n_ieq_constr=0,
            xl=0,
            xu=1,
            vtype=bool
        )
        self.evaluator = evaluator

    def _evaluate(self, x, out, *args, **kwargs):
        mask = np.array(x, dtype=int)
        out["F"] = self.evaluator.score_mask(mask)


def run_ga_feature_selection(
    X_train: np.ndarray,
    X_val: np.ndarray,
    y_train: np.ndarray,
    y_val: np.ndarray,
    feature_names: List[str],
    config: Config
):
    evaluator = FeatureSelectionEvaluator(
        X_train=X_train,
        X_val=X_val,
        y_train=y_train,
        y_val=y_val,
        classifier_name=config.classifier_name,
        alpha=config.alpha,
        random_state=config.random_state,
    )

    problem = GAFeatureSelectionProblem(evaluator)

    algorithm = GA(
        pop_size=config.ga_pop_size,
        sampling=BinaryRandomSampling(),
        crossover=HUX(),
        mutation=BitflipMutation(),
        eliminate_duplicates=True
    )

    start = time.time()

    result = minimize(
        problem,
        algorithm,
        termination=("n_gen", config.ga_generations),
        seed=config.random_state,
        verbose=False
    )

    elapsed = time.time() - start

    best_mask = np.array(result.X).astype(int).ravel()
    best_obj = float(result.F[0]) if np.ndim(result.F) > 0 else float(result.F)
    best_acc = evaluator.accuracy_of_mask(best_mask)

    selected_features = [feature_names[i] for i in range(len(best_mask)) if best_mask[i] == 1]

    return {
        "dataset": config.dataset_name,
        "classifier": config.classifier_name,
        "method": "GA",
        "n_total_features": len(feature_names),
        "n_selected": int(best_mask.sum()),
        "selected_ratio": float(best_mask.sum() / len(feature_names)),
        "objective": best_obj,
        "cv_accuracy": best_acc,
        "runtime_sec": elapsed,
        "selected_features": selected_features,
        "best_mask": best_mask,
    }

def continuous_to_binary_mask(x: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    mask = (np.asarray(x).ravel() >= threshold).astype(int)

    if mask.sum() == 0:
        mask[np.argmax(x)] = 1

    return mask


def run_binary_pso_feature_selection(
    X_train: np.ndarray,
    X_val: np.ndarray,
    y_train: np.ndarray,
    y_val: np.ndarray,
    feature_names: List[str],
    config: Config
):
    evaluator = FeatureSelectionEvaluator(
        X_train=X_train,
        X_val=X_val,
        y_train=y_train,
        y_val=y_val,
        classifier_name=config.classifier_name,
        alpha=config.alpha,
        random_state=config.random_state,
    )

    n_features = X_train.shape[1]
    lb = np.zeros(n_features)
    ub = np.ones(n_features)

    def objective_function(x):
        mask = continuous_to_binary_mask(x, threshold=config.pso_threshold)
        return evaluator.score_mask(mask)

    start = time.time()

    xopt, fopt = pso(
        objective_function,
        lb,
        ub,
        swarmsize=config.pso_swarmsize,
        maxiter=config.pso_maxiter,
        omega=config.pso_omega,
        phip=config.pso_phip,
        phig=config.pso_phig,
        debug=False
    )

    elapsed = time.time() - start

    best_mask = continuous_to_binary_mask(xopt, threshold=config.pso_threshold)
    best_acc = evaluator.accuracy_of_mask(best_mask)

    selected_features = [feature_names[i] for i in range(len(best_mask)) if best_mask[i] == 1]

    return {
        "dataset": config.dataset_name,
        "classifier": config.classifier_name,
        "method": "Binary PSO",
        "n_total_features": len(feature_names),
        "n_selected": int(best_mask.sum()),
        "selected_ratio": float(best_mask.sum() / len(feature_names)),
        "objective": float(fopt),
        "cv_accuracy": best_acc,
        "runtime_sec": elapsed,
        "selected_features": selected_features,
        "best_mask": best_mask,
    }


def run_baseline(
    X_train: np.ndarray,
    X_val: np.ndarray,
    y_train: np.ndarray,
    y_val: np.ndarray,
    feature_names: List[str],
    config: Config
):
    evaluator = FeatureSelectionEvaluator(
        X_train=X_train,
        X_val=X_val,
        y_train=y_train,
        y_val=y_val,
        classifier_name=config.classifier_name,
        alpha=config.alpha,
        random_state=config.random_state,
    )

    full_mask = np.ones(X_train.shape[1], dtype=int)

    start = time.time()
    full_acc = evaluator.accuracy_of_mask(full_mask)
    full_obj = evaluator.score_mask(full_mask)
    elapsed = time.time() - start

    return {
        "dataset": config.dataset_name,
        "classifier": config.classifier_name,
        "method": "Baseline",
        "n_total_features": len(feature_names),
        "n_selected": int(full_mask.sum()),
        "selected_ratio": 1.0,
        "objective": float(full_obj),
        "cv_accuracy": full_acc,
        "runtime_sec": elapsed,
        "selected_features": feature_names,
        "best_mask": full_mask,
    }


def run_experiment(dataset_name: str, classifier_name: str, base_config: Config):
    config = Config(
        dataset_name=dataset_name,
        classifier_name=classifier_name,
        random_state=base_config.random_state,
        alpha=base_config.alpha,
        ga_pop_size=base_config.ga_pop_size,
        ga_generations=base_config.ga_generations,
        pso_swarmsize=base_config.pso_swarmsize,
        pso_maxiter=base_config.pso_maxiter,
        pso_omega=base_config.pso_omega,
        pso_phip=base_config.pso_phip,
        pso_phig=base_config.pso_phig,
        pso_threshold=base_config.pso_threshold,
        sample_rows=base_config.sample_rows,
    )

    X, y, feature_names, raw_df = load_dataset(config)
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=config.random_state,
        stratify=y
    )

    print("=" * 90)
    print(f"DATASET: {dataset_name.upper()} | CLASSIFIER: {classifier_name.upper()}")
    print(f"Raw shape: {raw_df.shape} | Model shape: {X.shape} | Features: {len(feature_names)}")

    baseline_result = run_baseline(
        X_train, X_val, y_train, y_val, feature_names, config
    )
    print("ran baseline")
    ga_result = run_ga_feature_selection(
        X_train, X_val, y_train, y_val, feature_names, config
    )
    print("ran ga")

    pso_result = run_binary_pso_feature_selection(
        X_train, X_val, y_train, y_val, feature_names, config
    )
    print("ran binary pso")

    combo_results = [baseline_result, ga_result, pso_result]

    for r in combo_results:
        print(
            f"{r['method']:10s} | "
            f"Acc={r['cv_accuracy']:.6f} | "
            f"Selected={r['n_selected']}/{r['n_total_features']} | "
            f"Obj={r['objective']:.6f} | "
            f"Time={r['runtime_sec']:.2f}s"
        )

    return combo_results


def run_all():
    base_config = Config(
        random_state=42,
        alpha=0.99,
        ga_pop_size=20,
        ga_generations=10,
        pso_swarmsize=15,
        pso_maxiter=10,
        sample_rows=None
    )

    datasets = ["santander","census"]
    classifiers = ["svm","knn","dt"]

    all_results = []

    for dataset_name in datasets:
        for clf_name in classifiers:
            try:
                combo_results = run_experiment(dataset_name, clf_name, base_config)
                all_results.extend(combo_results)
            except Exception as e:
                print("=" * 90)
                print(f"FAILED: dataset={dataset_name}, classifier={clf_name}")
                print("Reason:", str(e))
                print()
                
    df_results = pd.DataFrame(all_results)

    if len(df_results) == 0:
        print("No results generated.")
        return None, None, None, None
        
    summary_cols = [
        "dataset",
        "classifier",
        "method",
        "cv_accuracy",
        "n_selected",
        "n_total_features",
        "selected_ratio",
        "objective",
        "runtime_sec",
    ]

    summary_table = df_results[summary_cols].copy()
    summary_table = summary_table.sort_values(
        by=["dataset", "classifier", "cv_accuracy"],
        ascending=[True, True, False]
    ).reset_index(drop=True)


    idx = df_results.groupby(["dataset", "classifier"])["cv_accuracy"].idxmax()
    best_table = df_results.loc[idx, summary_cols].copy()
    best_table = best_table.sort_values(
        by=["dataset", "classifier"]
    ).reset_index(drop=True)


    pivot_table = summary_table.pivot_table(
        index=["dataset", "classifier"],
        columns="method",
        values="cv_accuracy"
    ).reset_index()

    return df_results, summary_table, best_table, pivot_table


df_results, summary_table, best_table, pivot_table = run_all()

print("\n" + "=" * 100)
print("FINAL SUMMARY TABLE")
display(summary_table)

print("\n" + "=" * 100)
print("BEST METHOD FOR EACH DATASET + CLASSIFIER")
display(best_table)

print("\n" + "=" * 100)
print("ACCURACY PIVOT TABLE")
display(pivot_table)